In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import plotly.graph_objects as go
import warnings
import copy
warnings.filterwarnings('ignore')

# Path
PROJECT_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))
from utils.reproducibility_utils import seed_everything
from utils.data_loadingbins import load_data, load_test_data
from models.vanilla_rnn import VanillaRNN
from sklearn.metrics import mean_absolute_error

 # Load data
TRAIN_PATH = PROJECT_ROOT / "datasets_challenge" / "train"
TEST_PATH = PROJECT_ROOT / "datasets_challenge" / "test"
DIST_CSV = PROJECT_ROOT / "datasets_challenge" / "sideSlip_distribution.csv"
MODEL_DIR = PROJECT_ROOT / "expert_models"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(42) # Use your existing reproducibility util

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = VanillaRNN(
    input_size=22,          # number of features
    hidden_size=48,
    num_layers=1,
    dropout=0.0
).to(device)

criterion = nn.L1Loss()     # MAE (matches few-shot objective)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

In [3]:

def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for X, y in dataloader:
        X = X.to(device)
        y = y.to(device).unsqueeze(1)

        optimizer.zero_grad()
        preds = model(X)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


In [4]:
def validate(model, dataloader, device):
    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)

            pred = model(X).squeeze(1)
            preds.append(pred.cpu().numpy())
            targets.append(y.cpu().numpy())

    preds = np.concatenate(preds)
    targets = np.concatenate(targets)

    return mean_absolute_error(targets, preds)


In [5]:
train_loader, val_loader, pipeline = load_data(
    train_path=TRAIN_PATH,
    dist_csv_path=DIST_CSV,
    val_file_name="train_part_017.csv",
    columns_to_drop=["sideSlip", "vyCG", "time", "longitude", "latitude"],
    sequence_length=20,
    threshold=True,
    threshold_value=3.0,
    threshold_column="vxCG",
    target="sideSlip",
    batchsize=64
)

VALIDATION: Loading train_part_017.csv
Calculating weights for WeightedRandomSampler...
Max weight used: 7.0710678118654755
count    136559.000000
mean          1.309014
std           1.070870
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           7.071068
dtype: float64


In [6]:
#seed ensemble approach
SEEDS = [42, 1, 2, 3, 4, 123, 999, 2024, 7, 11]   
EPOCHS = 20

ensemble_models = []
ensemble_preds = []

def set_seed(seed):
    import random
    import numpy as np
    import torch

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [7]:
for seed in SEEDS:
    print(f"\n Training seed {seed}")
    
    set_seed(seed)
    
    model = VanillaRNN(
        input_size=22,          # number of features
        hidden_size=48,
        num_layers=1,
        dropout=0.0
    ).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
    
    best_val_mae = float("inf")
    best_model_state = None
    ensemble_states = []
    ensemble_preds = []
    
    for epoch in range(EPOCHS):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_mae = validate(model, val_loader, device)

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_model_state = copy.deepcopy(model.state_dict())

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"[Seed {seed}] "
            f"Epoch {epoch+1:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val MAE: {val_mae:.4f}"
            )   
    ensemble_states.append(best_model_state)
    


 Training seed 42
[Seed 42] Epoch 01 | Train Loss: 0.3882 | Val MAE: 0.3575
[Seed 42] Epoch 10 | Train Loss: 0.1815 | Val MAE: 0.2985
[Seed 42] Epoch 20 | Train Loss: 0.1664 | Val MAE: 0.3633

 Training seed 1
[Seed 1] Epoch 01 | Train Loss: 0.3732 | Val MAE: 0.3736
[Seed 1] Epoch 10 | Train Loss: 0.1826 | Val MAE: 0.3728
[Seed 1] Epoch 20 | Train Loss: 0.1670 | Val MAE: 0.3084

 Training seed 2
[Seed 2] Epoch 01 | Train Loss: 0.3798 | Val MAE: 0.3516
[Seed 2] Epoch 10 | Train Loss: 0.1801 | Val MAE: 0.3070
[Seed 2] Epoch 20 | Train Loss: 0.1658 | Val MAE: 0.2933

 Training seed 3
[Seed 3] Epoch 01 | Train Loss: 0.3881 | Val MAE: 0.2747
[Seed 3] Epoch 10 | Train Loss: 0.1801 | Val MAE: 0.3043
[Seed 3] Epoch 20 | Train Loss: 0.1672 | Val MAE: 0.3329

 Training seed 4
[Seed 4] Epoch 01 | Train Loss: 0.4120 | Val MAE: 0.3331
[Seed 4] Epoch 10 | Train Loss: 0.1771 | Val MAE: 0.3252
[Seed 4] Epoch 20 | Train Loss: 0.1614 | Val MAE: 0.3614

 Training seed 123
[Seed 123] Epoch 01 | Train Los

In [ ]:
targets = []
with torch.no_grad():
    for _, y in val_loader:
        targets.append(y.numpy())
targets = np.concatenate(targets)

# 🔹 ENSEMBLE PREDICTIONS
for state in ensemble_states:
    model = VanillaRNN(
        input_size=22,
        hidden_size=48,
        num_layers=1,
        dropout=0.0
    ).to(device)
    
    model.load_state_dict(state)
    model.eval()

    preds = []
    with torch.no_grad():
        for X, _ in val_loader:
            X = X.to(device)
            pred = model(X).squeeze(1)
            preds.append(pred.cpu().numpy())

    ensemble_preds.append(np.concatenate(preds))

ensemble_preds = np.mean(ensemble_preds, axis=0)
ensemble_mae = mean_absolute_error(targets, ensemble_preds)

print(f"\nSeed Ensemble Val MAE: {ensemble_mae:.4f}")



Seed Ensemble Val MAE: 0.2729
Models ensembled: 1
